# Speaking script · MTP speculative decoding in Rapid-LLM

- **Timing:** 12:00 of speaking, then 3:00 for Q&A.
- **How to use it:** Each bullet is written to say aloud. Pause briefly when pointing to a diagram, table, or code excerpt.


## Slide 1 · MTP speculative decoding in Rapid-LLM · 1:00

- Today I’ll explain how I extended Rapid-LLM to model multi-token prediction, or MTP, during speculative decoding.

- In ordinary autoregressive decoding, the backbone runs once to generate the next token. The model then uses that token as part of the context for the following step. That dependency makes ordinary decoding sequential.

- My question was whether a few inexpensive guesses could let one backbone pass advance several output tokens. I looked at two quantities: mean decode latency per output token and peak GPU memory.

- I modeled a MiMo-7B-shaped backbone on an A100 hardware configuration. I’ll start with how one speculative round works, then show what I changed in the simulator and where the speedup disappears.


## Slide 2 · Draft three tokens, verify four positions · 1:00

- Follow the top row first. There are three draft modules, and each proposes one token. Guess two depends on guess one, and guess three depends on the earlier draft, so these small steps run in sequence.

- Each module uses a projection from twice the hidden width to the hidden width, one transformer layer, and the shared vocabulary head.

- Now look at the lower row. The candidate tokens are already available, so the backbone can score four positions together in one causal pass. Those scores cover the three proposals and the position after the draft.

- If the second guess fails, we keep the first, emit the verifier’s replacement at the second position, and discard the third guess. If all three pass, the model can emit four tokens, including the verifier’s next token.


## Slide 3 · Acceptance controls round progress · 1:00

- The simulator takes the acceptance probabilities as inputs. The three values here are 0.9, 0.8, and 0.7, and each is conditional on all earlier guesses passing.

- Read the bars as the chance of reaching at least that many output tokens. The first bar is one because the verifier always supplies a token. To emit at least two, the first guess must pass. To emit at least three, the first two must pass, so we multiply 0.9 by 0.8.

- All three must pass to reach four outputs, giving 0.504. Adding these four probabilities gives 3.124 expected output tokens per round.

- That last number is an average across rounds. Individual rounds still emit whole tokens.


## Slide 4 · Where MTP changes the simulator · 1:00

- On the left is the YAML configuration I used. It requests three guesses, one module per guess, and one transformer layer per module. It also enables reuse of the backbone embedding and output head.

- The config parser checks that the module count matches the guess count and that there is one valid probability for each guess.

- The decode driver reuses Rapid-LLM’s sampled contexts. At each context it estimates the cost of a draft and verification round. Then it integrates the per-token estimate over the generation.

- The memory estimator adds the new module and cache allocations. The project uses model dimensions, hardware parameters, and supplied acceptance rates to estimate cost. There is no training or token sampling in this experiment.


## Slide 5 · Timing model for one round · 1:30

- I compute the two parts of the round separately. For the draft, I price each input projection, transformer block, and output-head call, then add the three module costs because the guesses run serially.

- For verification, I reuse the backbone decode graph with four query positions in one pass. The model prices attention as a rectangular operation using the longest context in the block. This is a modeling simplification for the verification workload.

- At the first sampled context, the draft costs about 2.98 milliseconds and verification costs about 12.33 milliseconds. Together they cost 15.31 milliseconds. One speculative round therefore costs more than one ordinary decode step.

- The Python excerpt shows where I add the draft time to the verifier time and divide by expected output tokens. Here, dividing 15.31 by 3.124 yields about 4.90 milliseconds per output token at that context.

- The simulator repeats this calculation at several context lengths and integrates over the generation. The benchmark slide reports that workload average, which is slightly higher.


## Slide 6 · GPU memory model · 1:00

- The memory model starts with the backbone’s existing allocations. Each draft module adds a projection with two times H squared weights, where H is the hidden width, plus the weights for its transformer layer.

- Each extra transformer layer also needs a KV cache. The model reserves a context-sized cache for each layer and space for the in-flight positions used during verification.

- The embedding and output head are shared with the backbone, so their weights don’t get duplicated in this configuration. The output projection still runs for each guess, so sharing saves weight memory while the timing model still charges the operation.

- The reported peak excludes draft temporary workspace. Draft-cache initialization time is also outside the timing model.


## Slide 7 · Workload and zero-guess check · 1:00

- For all the comparisons, I used the same dense MiMo-7B-shaped backbone, analytical A100 80 GB configuration, and batch size of one. Each run has a 1,024-token prompt and 1,024 generated tokens.

- The assignment’s correctness check was to set the number of guesses to zero. With that setting, there are no draft modules to price, one verification position, and one output token per decode step.

- I ran ordinary decoding with the MTP block omitted, then reran it with zero guesses explicitly set. The timing and memory reports matched byte for byte.

- That check establishes that enabling the configuration with zero guesses reproduces the existing decoder.


## Slide 8 · Three guesses cut decode latency by 60% · 1:30

- The top table gives the main comparison. Ordinary decode averaged 12.33 milliseconds per output token. Three MTP guesses brought that down to 4.92 milliseconds per token under the default acceptance probabilities.

- Dividing 12.33 by 4.92 gives about 2.51 times modeled decode speedup. That’s about a 60 percent reduction in decode latency per token for this workload.

- The other side of the comparison is memory. The per-GPU peak rose from 14.63 to 15.82 GiB, which is an increase of about 1.19 GiB. This is the peak across prefill and decode.

- The excerpt below the table is from the simulator’s results file. It shows the expected round progress and the reported latency and memory with more precision.

- The main reason for the speedup is that the round makes enough progress to spread the backbone and drafting costs over several output tokens. The next slide tests how sensitive that result is to the guess count and acceptance assumptions.


## Slide 9 · Guess count and acceptance ablations · 2:00

- The left table changes the number of guesses. For one guess I use the first acceptance probability. For two guesses I use the first two, and for three I use all three. Hardware, batch size, and token lengths stay fixed.

- Latency falls from 12.33 milliseconds per token at zero guesses to 7.02 with one, 5.48 with two, and 4.92 with three. Peak memory also rises at each step, from 14.63 GiB at zero guesses to 15.82 GiB at three.

- Notice that the incremental latency reduction gets smaller in this sweep. Later guesses help only when earlier ones pass. Adding a draft module still adds work and memory even when that guess contributes little to average round progress.

- The right table holds the number of guesses at three and changes the probabilities. Peak memory stays at 15.82 GiB because all three cases reserve the same module and cache capacity.

- At low acceptance, each conditional probability is 0.1. That yields just 1.111 expected outputs per round and 13.82 milliseconds per token, which is slower than ordinary decode.

- The default case yields 3.124 outputs and 4.92 milliseconds. Ideal acceptance yields four outputs and 3.84 milliseconds per token. The ideal row is a sensitivity case. It shows what the model predicts when every guess passes.


## Slide 10 · Break-even requires about 1.25 tokens per round · 1:00

- The break-even calculation explains why the low-acceptance case loses. Take the time for one full speculative round and divide by the time for one ordinary decode step at the same context.

- At the first sampled context that is about 15.31 milliseconds divided by 12.29 milliseconds, or roughly 1.25 tokens per round. The round has to advance more than that on average to improve latency.

- The default case clears the threshold at 3.124 expected tokens. The low-acceptance case falls short at 1.111.

- My conclusion is that the default configuration gives a 2.51 times modeled decode speedup for about 1.19 GiB of extra peak memory, and that benefit depends on the acceptance rates. Thanks. I’m happy to take questions.


## Q&A rehearsal

- **Why can the verifier score all the positions at once?**
  - Every draft token in the candidate block is already available as input. Causal masking makes each prediction depend only on its preceding tokens.

- **Why use 3.124 instead of the average of the acceptance rates?**
  - 3.124 measures expected outputs per round. Each additional output is earned only if all guesses before it pass, so the calculation uses products of conditional probabilities.

- **Did you measure this on a physical A100?**
  - The numbers come from Rapid-LLM’s analytical A100 configuration. They estimate single-GPU latency and memory for the workload on slide 7.

- **Why does the shared output head still cost time?**
  - The modules reuse the existing weight tensor, but each head invocation still computes the vocabulary projection.

- **Where are the biggest modeling approximations?**
  - Verification uses rectangular attention at the longest context in the block. Round progress uses a steady-state expectation, and the draft cache assumes full context. Draft-cache initialization time and temporary workspace are outside the model.
